# ReadmitIQ - Clinical Note Embeddings on Colab GPU

This self-contained notebook embeds the synthetic clinical notes for all three splits (`train`, `val`, `test`) using `all-MiniLM-L6-v2` on a free T4 GPU in ~3-5 minutes.

### Instructions
1. In Google Colab, change runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Upload the 3 parquet files to the Colab working directory:
   - `train.parquet`
   - `val.parquet`
   - `test.parquet`
3. Run all cells.
4. Download the generated output files:
   - `embeddings_train.npy`
   - `embeddings_val.npy`
   - `embeddings_test.npy`
   - `embeddings_checksum.json`
5. Place them into `data/processed/` in the local repository.

In [ ]:
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
import hashlib
import json
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')
assert device == 'cuda', 'GPU recommended for fast embedding! Go to Runtime > Change runtime type > T4 GPU'

model_name = 'all-MiniLM-L6-v2'
print(f'Loading {model_name} onto {device}...')
model = SentenceTransformer(model_name, device=device)

In [ ]:
def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

splits = ['train', 'val', 'test']
metadata = {}
chunk_size = 10000

for split in splits:
    parquet_file = f'{split}.parquet'
    assert Path(parquet_file).exists(), f'Missing {parquet_file}! Please upload it.'
    df = pd.read_parquet(parquet_file)
    assert 'clinical_note' in df.columns, f'clinical_note column missing from {parquet_file}!'
    notes = df['clinical_note'].tolist()
    n_rows = len(notes)
    out_file = f'embeddings_{split}.npy'
    checkpoint_dir = Path(f'checkpoints_{split}')
    checkpoint_dir.mkdir(exist_ok=True)
    
    # Split-level checkpoint: if already completed and valid, skip
    if Path(out_file).exists():
        existing_arr = np.load(out_file, mmap_mode='r')
        if existing_arr.shape == (n_rows, 384):
            print(f'[{split.upper()}] Checkpoint found! {out_file} already exists ({n_rows:,} rows). Skipping encoding.')
            metadata[split] = {
                'rows': n_rows,
                'embedding_dim': 384,
                'file': out_file,
                'sha256': compute_sha256(out_file),
                'seconds': 0.0,
                'status': 'loaded_from_checkpoint'
            }
            continue
            
    print(f'[{split.upper()}] Encoding {n_rows:,} notes with checkpoint chunk_size={chunk_size:,}, batch_size=256...')
    t0 = time.time()
    chunks = []
    
    for start_i in range(0, n_rows, chunk_size):
        end_i = min(start_i + chunk_size, n_rows)
        chunk_path = checkpoint_dir / f'chunk_{start_i}_{end_i}.npy'
        
        # Chunk-level checkpoint: if this chunk was already computed before interruption, reuse it!
        if chunk_path.exists():
            c_arr = np.load(chunk_path)
            if c_arr.shape == (end_i - start_i, 384):
                print(f'  ✓ Loaded checkpoint chunk {start_i:,}:{end_i:,} ({end_i - start_i:,} rows) from disk.')
                chunks.append(c_arr)
                continue
                
        print(f'  -> Computing chunk {start_i:,}:{end_i:,} ({end_i - start_i:,} notes)...')
        sub_notes = notes[start_i:end_i]
        c_arr = model.encode(
            sub_notes,
            batch_size=256,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)
        
        # Save chunk checkpoint immediately to disk
        np.save(chunk_path, c_arr)
        chunks.append(c_arr)
        print(f'  ✓ Checkpoint saved: {chunk_path.name}')
        
    embeddings = np.vstack(chunks)
    elapsed = time.time() - t0
    
    assert embeddings.shape == (n_rows, 384), f'Expected ({n_rows}, 384), got {embeddings.shape}'
    np.save(out_file, embeddings)
    print(f'[{split.upper()}] Saved full split to {out_file} ({n_rows:,} rows) in {elapsed:.1f}s.\n')
    
    metadata[split] = {
        'rows': n_rows,
        'embedding_dim': 384,
        'file': out_file,
        'sha256': compute_sha256(out_file),
        'seconds': round(elapsed, 2),
        'status': 'computed_with_checkpoints'
    }

with open('embeddings_checksum.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Encoding complete! Metadata saved to embeddings_checksum.json.')


In [ ]:
# Verification check
with open('embeddings_checksum.json') as f:
    meta = json.load(f)
print(json.dumps(meta, indent=2))

# Zip for one-click download
!zip -j readmit_embeddings.zip embeddings_train.npy embeddings_val.npy embeddings_test.npy embeddings_checksum.json
print('Ready! Download readmit_embeddings.zip from the Colab file browser.')